In [3]:
import pandas as pd
import pymongo
import logging
import sys
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# 1. Setup Logging (Rubric Requirement)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.FileHandler("ingestion.log"), logging.StreamHandler(sys.stdout)]
)

def ingest_power_plants(csv_path):
    # Get URI from environment variable
    mongo_uri = os.getenv("MONGO_URI")
    
    if not mongo_uri:
        logging.error("MONGO_URI not found in .env file!")
        return

    try:
        # 2. Load Data
        logging.info("Reading CSV file...")
        df = pd.read_csv(csv_path)

        # 3. Data Cleaning & Structuring (The "Data Creation" step)
        logging.info("Cleaning and restructuring data...")
        df = df.replace('[null]', 'Unknown').fillna('Unknown')

        # Convert to Nested Document Model (Implicit Schema)
        documents = []
        for _, row in df.iterrows():
            doc = {
                "gppd_idnr": row['gppd_idnr'],
                "name": row['Powerplant Name'],
                "location": {
                    "country": row['Country'],
                    "latitude": float(row['Latitude']),
                    "longitude": float(row['Longitude'])
                },
                "specs": {
                    "primary_fuel": row['Primary Fuel'],
                    "capacity_mw": float(row['Capacity (MW)'])
                },
                "provenance": {
                    "owner": row['Owner'],
                    "source": row['Source']
                }
            }
            documents.append(doc)

        # 4. Connect to MongoDB Atlas
        logging.info("Connecting to MongoDB Atlas...")
        client = pymongo.MongoClient(mongo_uri)
        db = client['power_grid_db']
        collection = db['plants']

        # 5. Insert Data
        logging.info(f"Inserting {len(documents)} documents...")
        collection.delete_many({}) 
        result = collection.insert_many(documents)
        
        logging.info(f"Successfully inserted {len(result.inserted_ids)} documents.")

    except Exception as e:
        logging.error(f"An unexpected error occurred: {e}")
    finally:
        if 'client' in locals():
            client.close()
            logging.info("MongoDB connection closed.")

if __name__ == "__main__":
    # Ensure this path is correct relative to your notebook
    CSV_FILE = "../data/GlobalPowerPlant.csv"
    ingest_power_plants(CSV_FILE)

2026-04-28 13:29:50,725 [INFO] Reading CSV file...
2026-04-28 13:29:50,904 [INFO] Cleaning and restructuring data...
2026-04-28 13:29:51,613 [INFO] Connecting to MongoDB Atlas...
2026-04-28 13:29:51,672 [INFO] Inserting 29910 documents...
2026-04-28 13:30:02,696 [INFO] Successfully inserted 29910 documents.
2026-04-28 13:30:02,717 [INFO] MongoDB connection closed.
